# DLQI 量化交易模型训练 (Colab GPU)

**A 股数据 + 深度学习模型训练**

**步骤:** 运行时 → 更改运行时类型 → T4 GPU → 全部运行

**股票池:** 贵州茅台(600519)、中国平安(601318)、招商银行(600036)、宁德时代(300750)、比亚迪(002594)

**数据源:** AKShare（A 股前复权日线，10 年历史数据）

## 1. 安装依赖

In [ ]:
# 安装依赖
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q lightgbm xgboost scikit-learn pandas numpy joblib
!pip install -q akshare

print("✅ 依赖安装完成")

## 2. 检查 GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\u2705 GPU 可用: {gpu_name} ({gpu_memory:.1f} GB)")
    DEVICE = torch.device('cuda')
else:
    print("\u26a0\ufe0f GPU 不可用, 使用 CPU")
    DEVICE = torch.device('cpu')

## 3. 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = '/content/drive/MyDrive/DLQI'
MODELS_DIR = f'{BASE_DIR}/models'
TASKS_DIR = f'{BASE_DIR}/tasks'
RESULTS_DIR = f'{BASE_DIR}/results'
DATA_DIR = f'{BASE_DIR}/data/raw'

for d in [MODELS_DIR, TASKS_DIR, RESULTS_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

# 检查数据
csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
print(f"\u2705 Drive 已挂载, 发现 {len(csv_files)} 个数据文件:")
for f in sorted(csv_files):
    print(f"   {f}")

## 4. 模型定义

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import json, time, traceback, joblib, glob, os
import lightgbm as lgb
import xgboost as xgb


class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, dropout=dropout if num_layers > 1 else 0,
            batch_first=True, bidirectional=True
        )
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size * 2, num_heads=4, dropout=dropout, batch_first=True
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        return self.fc(attn_out[:, -1, :])


class TransformerModel(nn.Module):
    """改进版 Transformer: 正弦位置编码 + 因果掩码 + 更小模型"""
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.d_model = d_model
        self.input_proj = nn.Linear(input_size, d_model)
        # 正弦位置编码（不需要学习，小数据更稳定）
        pe = torch.zeros(500, d_model)
        position = torch.arange(0, 500, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
    def forward(self, x):
        seq_len = x.size(1)
        x = self.input_proj(x)
        x = x + self.pe[:, :seq_len, :]
        # 因果掩码：防止未来信息泄露
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
        x = self.transformer(x, mask=mask)
        return self.fc(x[:, -1, :])


print("✅ 模型定义完成")
print(f"   TransformerModel 参数量: {sum(p.numel() for p in TransformerModel(12).parameters()):,}")

## 5. 训练器

In [ ]:
import os, json, time, traceback, joblib, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from datetime import datetime

# 确保关键变量存在
if 'DEVICE' not in dir():
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"   DEVICE 自动设置: {DEVICE}")
if 'BASE_DIR' not in dir():
    BASE_DIR = '/content/drive/MyDrive/DLQI'
    MODELS_DIR = f'{BASE_DIR}/models'
    TASKS_DIR = f'{BASE_DIR}/tasks'
    RESULTS_DIR = f'{BASE_DIR}/results'
    DATA_DIR = f'{BASE_DIR}/data/raw'
    for d in [MODELS_DIR, TASKS_DIR, RESULTS_DIR, DATA_DIR]:
        os.makedirs(d, exist_ok=True)
    print(f"   路径自动设置: {BASE_DIR}")

class ModelTrainer:
    def __init__(self):
        self.device = DEVICE
        self.scaler = StandardScaler()

    def load_local_data(self, symbol):
        for prefix in ['cn_', '']:
            path = os.path.join(DATA_DIR, f'{prefix}{symbol}.csv')
            if os.path.exists(path):
                df = pd.read_csv(path, parse_dates=['date'])
                print(f"   从本地加载: {path} ({len(df)} 条)")
                return df
        raise FileNotFoundError(f"未找到 {symbol} 的数据文件")

    def fetch_data(self, symbol, start_date, end_date=None):
        try:
            return self.load_local_data(symbol)
        except FileNotFoundError:
            print(f"   本地无数据, 用 akshare 下载 {symbol}...")
            import akshare as ak
            df = yf.Ticker(symbol).history(start=start_date, end=end_date, auto_adjust=True)
            df = df.reset_index().rename(columns={
                'Date': 'date', 'Open': 'open', 'High': 'high',
                'Low': 'low', 'Close': 'close', 'Volume': 'volume'
            })
            df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)
            return df

    def prepare_features(self, df, sequence_length=60):
        df = df.copy()
        df['returns'] = df['close'].pct_change()
        df['ma_5'] = df['close'].rolling(5).mean()
        df['ma_20'] = df['close'].rolling(20).mean()
        df['volatility'] = df['returns'].rolling(20).std()
        delta = df['close'].diff()
        gain = delta.where(delta > 0, 0).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        df['rsi'] = 100 - (100 / (1 + gain / (loss + 1e-10)))
        ema12 = df['close'].ewm(span=12).mean()
        ema26 = df['close'].ewm(span=26).mean()
        df['macd'] = ema12 - ema26
        df['macd_signal'] = df['macd'].ewm(span=9).mean()
        df = df.dropna()
        df['target'] = df['returns'].shift(-1)
        df = df.dropna()
        feature_cols = ['open','high','low','close','volume',
                       'returns','ma_5','ma_20','volatility','rsi','macd','macd_signal']
        X = df[feature_cols].values
        y = df['target'].values
        X = self.scaler.fit_transform(X)
        X_seq, y_seq = [], []
        for i in range(sequence_length, len(X)):
            X_seq.append(X[i-sequence_length:i])
            y_seq.append(y[i])
        return np.array(X_seq), np.array(y_seq), feature_cols

    def prepare_features_raw(self, df):
        df = df.copy()
        df['returns'] = df['close'].pct_change()
        df['ma_5'] = df['close'].rolling(5).mean()
        df['ma_20'] = df['close'].rolling(20).mean()
        df['volatility'] = df['returns'].rolling(20).std()
        delta = df['close'].diff()
        gain = delta.where(delta > 0, 0).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        df['rsi'] = 100 - (100 / (1 + gain / (loss + 1e-10)))
        ema12 = df['close'].ewm(span=12).mean()
        ema26 = df['close'].ewm(span=26).mean()
        df['macd'] = ema12 - ema26
        df['macd_signal'] = df['macd'].ewm(span=9).mean()
        df = df.dropna()
        df['target'] = df['returns'].shift(-1)
        df = df.dropna()
        feature_cols = ['open','high','low','close','volume',
                       'returns','ma_5','ma_20','volatility','rsi','macd','macd_signal']
        return df[feature_cols].values, df['target'].values, feature_cols

    def train_lstm(self, X, y, epochs=100, batch_size=32, lr=0.001, cb=None):
        split = int(len(X) * 0.8)
        train_loader = DataLoader(TimeSeriesDataset(X[:split], y[:split]), batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(TimeSeriesDataset(X[split:], y[split:]), batch_size=batch_size)
        model = LSTMModel(input_size=X.shape[2]).to(self.device)
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        best_loss, best_state, patience = float('inf'), None, 0
        for epoch in range(epochs):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                optimizer.zero_grad()
                loss = criterion(model(xb).squeeze(), yb)
                loss.backward()
                optimizer.step()
            model.eval()
            vl = 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(self.device), yb.to(self.device)
                    vl += criterion(model(xb).squeeze(), yb).item()
            vl /= len(val_loader)
            if vl < best_loss:
                best_loss, best_state, patience = vl, model.state_dict().copy(), 0
            else:
                patience += 1
            if cb: cb((epoch+1)/epochs, f"Epoch {epoch+1}/{epochs}, Val Loss: {vl:.6f}")
            if (epoch+1) % 10 == 0: print(f"   Epoch {epoch+1}/{epochs} - Val Loss: {vl:.6f}")
            if patience >= 10: print(f"   Early stopping at epoch {epoch+1}"); break
        model.load_state_dict(best_state)
        return model, best_loss

    def train_transformer(self, X, y, epochs=100, batch_size=128, lr=0.001, cb=None):
        split = int(len(X) * 0.8)
        train_loader = DataLoader(TimeSeriesDataset(X[:split], y[:split]), batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(TimeSeriesDataset(X[split:], y[split:]), batch_size=batch_size)
        model = TransformerModel(input_size=X.shape[2]).to(self.device)
        criterion = nn.MSELoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        best_loss, best_state, patience_cnt = float('inf'), None, 0
        for epoch in range(epochs):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                optimizer.zero_grad()
                loss = criterion(model(xb).squeeze(), yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            scheduler.step()
            model.eval()
            vl = 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(self.device), yb.to(self.device)
                    vl += criterion(model(xb).squeeze(), yb).item()
            vl /= len(val_loader)
            if vl < best_loss:
                best_loss, best_state, patience_cnt = vl, model.state_dict().copy(), 0
            else:
                patience_cnt += 1
            if cb: cb((epoch+1)/epochs, f"Epoch {epoch+1}/{epochs}, Val Loss: {vl:.6f}")
            if (epoch+1) % 10 == 0: print(f"   Epoch {epoch+1}/{epochs} - Val Loss: {vl:.6f}")
            if patience_cnt >= 15: print(f"   Early stopping at epoch {epoch+1}"); break
        model.load_state_dict(best_state)
        return model, best_loss

    def train_lightgbm(self, X, y, cb=None):
        import lightgbm as lgb
        X_flat = X.reshape(X.shape[0], -1)
        split = int(len(X_flat) * 0.8)
        train_data = lgb.Dataset(X_flat[:split], label=y[:split])
        val_data = lgb.Dataset(X_flat[split:], label=y[split:], reference=train_data)
        params = {'objective': 'regression', 'metric': 'mse', 'learning_rate': 0.05, 'num_leaves': 31, 'verbose': -1}
        model = lgb.train(params, train_data, num_boost_round=500, valid_sets=[val_data], callbacks=[lgb.early_stopping(50)])
        val_pred = model.predict(X_flat[split:])
        val_loss = np.mean((val_pred - y[split:]) ** 2)
        if cb: cb(1.0, f"Val Loss: {val_loss:.6f}")
        return model, val_loss

    def train_xgboost(self, X, y, cb=None):
        import xgboost as xgb
        X_flat = X.reshape(X.shape[0], -1)
        split = int(len(X_flat) * 0.8)
        dtrain = xgb.DMatrix(X_flat[:split], label=y[:split])
        dval = xgb.DMatrix(X_flat[split:], label=y[split:])
        params = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'learning_rate': 0.05, 'max_depth': 6}
        if torch.cuda.is_available(): params['device'] = 'cuda'
        model = xgb.train(params, dtrain, num_boost_round=500, evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
        val_pred = model.predict(dval)
        val_loss = np.mean((val_pred - y[split:]) ** 2)
        if cb: cb(1.0, f"Val Loss: {val_loss:.6f}")
        return model, val_loss

    def save_model(self, model, model_type, symbol, metadata=None):
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        model_id = f"{model_type}_{symbol}_{timestamp}"
        model_path = os.path.join(MODELS_DIR, model_id)
        os.makedirs(model_path, exist_ok=True)
        if model_type in ['lstm', 'transformer']:
            torch.save(model.state_dict(), os.path.join(model_path, 'model.pt'))
        elif model_type == 'lightgbm':
            model.save_model(os.path.join(model_path, 'model.txt'))
        elif model_type == 'xgboost':
            model.save_model(os.path.join(model_path, 'model.json'))
        joblib.dump(self.scaler, os.path.join(model_path, 'scaler.pkl'))
        meta = {'model_id': model_id, 'model_type': model_type, 'symbol': symbol,
                'created_at': timestamp, **(metadata or {})}
        with open(os.path.join(model_path, 'metadata.json'), 'w') as f:
            json.dump(meta, f, indent=2)
        return model_id

trainer = ModelTrainer()
print("✅ 训练器初始化完成")

## 6. 下载 A 股数据

使用 AKShare 下载 5 只 A 股 10 年前复权日线数据

In [ ]:
import akshare as ak
import time as _time

CN_STOCKS = {
    "600519": "贵州茅台",
    "601318": "中国平安",
    "600036": "招商银行",
    "300750": "宁德时代",
    "002594": "比亚迪",
}

raw_dir = os.path.join(BASE_DIR, "data", "raw")
os.makedirs(raw_dir, exist_ok=True)

for symbol, name in CN_STOCKS.items():
    csv_path = os.path.join(raw_dir, f"cn_{symbol}.csv")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(f"✅ {symbol} ({name}) 已存在: {len(df)} 条")
        continue
    try:
        df = ak.stock_zh_a_hist(
            symbol=symbol, period="daily",
            start_date="20160101", adjust="qfq"
        )
        df = df.rename(columns={
            "日期": "date", "开盘": "open", "最高": "high",
            "最低": "low", "收盘": "close", "成交量": "volume",
        })
        df = df[["date", "open", "high", "low", "close", "volume"]]
        df.to_csv(csv_path, index=False)
        print(f"✅ {symbol} ({name}): {len(df)} 条")
        _time.sleep(1)
    except Exception as e:
        print(f"❌ {symbol} ({name}): {e}")

print(f"\n数据保存在: {raw_dir}")

## 7. 多股票联合训练 Transformer

用 5 只 A 股数据联合训练分类版 Transformer

In [ ]:
import glob, os, gc

class LazySeqDataset(Dataset):
    """按需生成序列"""
    def __init__(self, stock_arrays, seq_len=60):
        self.seq_len = seq_len
        self.stocks = stock_arrays
        self.index_map = []
        for si, (X, y) in enumerate(self.stocks):
            for j in range(seq_len, len(X)):
                self.index_map.append((si, j))

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        si, j = self.index_map[idx]
        X, y = self.stocks[si]
        seq = X[j - self.seq_len:j]
        target = y[j]
        return torch.FloatTensor(seq), torch.LongTensor([target]).squeeze()


class TransformerClassifier(nn.Module):
    """分类版 Transformer：预测涨(1)/跌(0)"""
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        pe = torch.zeros(500, d_model)
        position = torch.arange(0, 500, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 2),  # 2 类：跌(0), 涨(1)
        )

    def forward(self, x):
        seq_len = x.size(1)
        x = self.input_proj(x)
        x = x + self.pe[:, :seq_len, :]
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
        x = self.transformer(x, mask=mask)
        return self.fc(x[:, -1, :])  # logits, shape (batch, 2)


def train_multi_stock_transformer(epochs=50, seq_len=60, batch_size=256, max_stocks=None):
    """
    多股票联合训练 Transformer（分类版）
    预测涨/跌方向，用 CrossEntropyLoss，彻底避免模型坍缩
    """
    feature_cols = ['open','high','low','close','volume',
                    'returns','ma_5','ma_20','volatility','rsi','macd','macd_signal']

    # 1. 收集数据
    csv_files = sorted(glob.glob(os.path.join(DATA_DIR, 'cn_*.csv')))
    csv_files = [f for f in csv_files if 'idx_' not in os.path.basename(f)]
    if max_stocks: csv_files = csv_files[:max_stocks]
    print(f"📊 找到 {len(csv_files)} 只股票数据文件")

    # 2. 加载
    raw_data = []
    for i, path in enumerate(csv_files):
        try:
            df = pd.read_csv(path)
            col_map = {'Date':'date','Open':'open','High':'high','Low':'low','Close':'close','Volume':'volume'}
            df = df.rename(columns={k:v for k,v in col_map.items() if k in df.columns})
            X_raw, y_raw, _ = trainer.prepare_features_raw(df)
            if len(X_raw) >= seq_len + 50:
                # 转为分类标签：涨=1, 跌=0
                y_cls = (y_raw > 0).astype(np.int64)
                raw_data.append((X_raw, y_cls))
        except: pass
        if (i+1) % 50 == 0: print(f"   加载: {i+1}/{len(csv_files)}, 有效: {len(raw_data)}")

    total_rows = sum(len(X) for X, _ in raw_data)
    print(f"✅ 有效股票: {len(raw_data)}, 总数据量: {total_rows:,}")

    # 3. fit scaler
    train_chunks = [X[:int(len(X)*0.7)] for X, _ in raw_data]
    feature_scaler = StandardScaler()
    feature_scaler.fit(np.concatenate(train_chunks, axis=0))
    del train_chunks; gc.collect()

    # 统计类别分布
    all_labels = np.concatenate([y[:int(len(y)*0.7)] for _, y in raw_data])
    pos_ratio = all_labels.mean()
    print(f"   类别分布: 涨 {pos_ratio*100:.1f}% / 跌 {(1-pos_ratio)*100:.1f}%")

    # 4. 分割 + 标准化
    train_stocks, val_stocks = [], []
    for X_raw, y_cls in raw_data:
        te = int(len(X_raw) * 0.7)
        ve = int(len(X_raw) * 0.85)
        train_stocks.append((feature_scaler.transform(X_raw[:te]).astype(np.float32), y_cls[:te]))
        val_stocks.append((feature_scaler.transform(X_raw[te:ve]).astype(np.float32), y_cls[te:ve]))
    del raw_data; gc.collect()

    # 5. Dataset
    train_ds = LazySeqDataset(train_stocks, seq_len)
    val_ds = LazySeqDataset(val_stocks, seq_len)
    print(f"   训练样本: {len(train_ds):,}, 验证样本: {len(val_ds):,}")

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    # 6. 训练
    print(f"\n🚀 开始训练 (epochs={epochs}, batch={batch_size}, 分类任务)")
    model = TransformerClassifier(input_size=12).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    warmup_epochs = 3
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / (epochs - warmup_epochs)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # 加权 CrossEntropy 处理类别不平衡
    weight = torch.FloatTensor([pos_ratio, 1 - pos_ratio]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weight)

    best_acc, best_loss, best_state, patience_cnt = 0, float('inf'), None, 0
    for epoch in range(epochs):
        model.train()
        epoch_loss, epoch_correct, epoch_total = 0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
            epoch_correct += (logits.argmax(1) == yb).sum().item()
            epoch_total += len(yb)
        scheduler.step()
        train_acc = epoch_correct / epoch_total

        # 验证
        model.eval()
        val_correct, val_total, val_loss_sum = 0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                val_loss_sum += criterion(logits, yb).item() * len(yb)
                val_correct += (logits.argmax(1) == yb).sum().item()
                val_total += len(yb)
        val_acc = val_correct / val_total
        val_loss = val_loss_sum / val_total

        if val_acc > best_acc:
            best_acc = val_acc
            best_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1

        if (epoch+1) % 5 == 0 or patience_cnt == 0:
            print(f"   Epoch {epoch+1:3d}/{epochs} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val Loss: {val_loss:.4f} | Best: {best_acc:.4f} | P: {patience_cnt}")
        if patience_cnt >= 15:
            print(f"   ⏹ Early stopping at epoch {epoch+1}"); break

    model.load_state_dict(best_state)
    print(f"\n📈 最终验证集准确率: {best_acc:.4f}")

    # 保存
    trainer.scaler = feature_scaler
    model_id = trainer.save_model(model, 'transformer', 'MULTI', {
        'val_loss': float(best_loss), 'direction_accuracy': float(best_acc),
        'n_stocks': len(csv_files), 'data_points': len(train_ds) + len(val_ds),
        'epochs': epochs, 'features': feature_cols,
        'task_type': 'classification',
    })
    print(f"✅ 模型已保存: {model_id}")
    return model_id, best_loss, best_acc

# 运行
model_id, loss, acc = train_multi_stock_transformer(epochs=50)
print(f"\n🎯 最终结果: model={model_id}, loss={loss:.6f}, dir_acc={acc:.4f}")

## 8. 任务处理器

In [ ]:
class TaskProcessor:
    def __init__(self):
        self.processed = set()
        pf = os.path.join(RESULTS_DIR, '.processed')
        if os.path.exists(pf):
            with open(pf) as f: self.processed = set(f.read().splitlines())

    def save_processed(self):
        with open(os.path.join(RESULTS_DIR, '.processed'), 'w') as f:
            f.write('\n'.join(self.processed))

    def update_status(self, tid, status, progress=0, msg=None, result=None, error=None):
        with open(os.path.join(RESULTS_DIR, f"{tid}_status.json"), 'w') as f:
            json.dump({'task_id': tid, 'status': status, 'progress': progress,
                       'message': msg, 'result': result, 'error': error,
                       'updated_at': datetime.now().isoformat()}, f, indent=2)

    def process(self, task):
        tid = task.get('task_id', task['_fn'].replace('.json',''))
        task_type = task.get('type', 'train')
        print(f"\n{'='*50}\n📋 处理任务: {tid} (类型: {task_type})")

        try:
            # 多股票联合训练
            if task_type == 'train_multi':
                self._process_multi(task, tid)
            # 批量下载数据
            elif task_type == 'download_cn':
                self._process_download(task, tid)
            # 单股票训练（原有逻辑）
            else:
                self._process_single(task, tid)
        except Exception as e:
            self.update_status(tid, 'failed', error=str(e))
            print(f"❌ 失败: {e}")
            traceback.print_exc()

        self.processed.add(task['_fn'])
        self.save_processed()

    def _process_single(self, task, tid):
        """单股票训练"""
        self.update_status(tid, 'running', 0.1, '开始...')
        symbol = task['symbol']
        model_type = task.get('model_type', 'lstm')
        epochs = task.get('epochs', 100)

        self.update_status(tid, 'running', 0.1, '获取数据...')
        df = trainer.fetch_data(symbol, task.get('start_date','2020-01-01'), task.get('end_date'))
        print(f"   数据: {len(df)} 条")

        self.update_status(tid, 'running', 0.15, '准备特征...')
        X, y, features = trainer.prepare_features(df)
        print(f"   特征: {X.shape}")

        self.update_status(tid, 'running', 0.2, '训练中...')
        def cb(p, m): self.update_status(tid, 'running', 0.2 + p*0.7, m)

        if model_type == 'lstm': model, loss = trainer.train_lstm(X, y, epochs=epochs, cb=cb)
        elif model_type == 'transformer': model, loss = trainer.train_transformer(X, y, epochs=epochs, cb=cb)
        elif model_type == 'lightgbm': model, loss = trainer.train_lightgbm(X, y, cb=cb)
        elif model_type == 'xgboost': model, loss = trainer.train_xgboost(X, y, cb=cb)
        else: raise ValueError(f"未知模型: {model_type}")

        self.update_status(tid, 'running', 0.95, '保存模型...')
        mid = trainer.save_model(model, model_type, symbol, {
            'task_id': tid, 'val_loss': float(loss), 'epochs': epochs,
            'features': features, 'data_points': len(df)
        })
        result = {'model_id': mid, 'val_loss': float(loss)}
        self.update_status(tid, 'completed', 1.0, '完成', result)
        print(f"✅ 完成! 模型: {mid}, Loss: {loss:.6f}")

    def _process_multi(self, task, tid):
        """多股票联合训练 Transformer"""
        epochs = task.get('epochs', 30)
        max_stocks = task.get('max_stocks', None)
        self.update_status(tid, 'running', 0.05, '启动多股票联合训练...')

        # 先检查是否需要下载数据
        csv_files = glob.glob(os.path.join(DATA_DIR, 'cn_*.csv'))
        csv_files = [f for f in csv_files if 'idx_' not in os.path.basename(f)]
        if len(csv_files) < 20:
            self.update_status(tid, 'running', 0.05, f'数据不足({len(csv_files)}只), 先下载 A 股...')
            download_cn_data(start=task.get('start_date', '2016-01-01'), delay=0.3)

        def cb(p, m): self.update_status(tid, 'running', 0.1 + p*0.85, m)

        model_id, loss, dir_acc = train_multi_stock_transformer(
            epochs=epochs, max_stocks=max_stocks
        )
        result = {'model_id': model_id, 'val_loss': float(loss), 'direction_accuracy': dir_acc}
        self.update_status(tid, 'completed', 1.0, '多股票训练完成', result)
        print(f"✅ 完成! 模型: {model_id}, Loss: {loss:.6f}, Dir Acc: {dir_acc:.4f}")

    def _process_download(self, task, tid):
        """批量下载 A 股 数据"""
        self.update_status(tid, 'running', 0.1, '开始下载 A 股 数据...')
        start = task.get('start_date', '2016-01-01')
        download_cn_data(start=start, delay=task.get('delay', 0.3))
        csv_count = len([f for f in os.listdir(DATA_DIR) if f.startswith('cn_') and f.endswith('.csv') and 'idx' not in f])
        self.update_status(tid, 'completed', 1.0, f'下载完成: {csv_count} 只股票', {'n_stocks': csv_count})
        print(f"✅ 下载完成: {csv_count} 只股票")

    def run_loop(self, interval=30):
        print(f"\n🔄 开始监控任务 (每 {interval}s)")
        print(f"   任务目录: {TASKS_DIR}")
        print(f"   支持类型: train(单股票), train_multi(多股票联合), download_cn(下载数据)")
        print(f"   提交方式: 将 JSON 文件放入上述目录\n")
        while True:
            try:
                for fn in os.listdir(TASKS_DIR):
                    if fn.endswith('.json') and fn not in self.processed:
                        with open(os.path.join(TASKS_DIR, fn)) as f:
                            task = json.load(f)
                        task['_fn'] = fn
                        self.process(task)
                print(f"⏳ {datetime.now().strftime('%H:%M:%S')} 等待新任务...", end='\r')
                time.sleep(interval)
            except KeyboardInterrupt:
                print("\n👋 停止"); break
            except Exception as e:
                print(f"⚠️ 错误: {e}"); time.sleep(interval)


processor = TaskProcessor()
print("✅ 任务处理器就绪")
print("   支持任务类型:")
print("   - train: 单股票训练 {symbol, model_type, epochs}")
print("   - train_multi: 多股票联合训练 {epochs, max_stocks?}")
print("   - download_cn: 批量下载数据 {start_date?}")

## 9. 启动自动监控

运行下面的单元格后, Colab 会每 30 秒检查一次任务文件夹.

从本地提交任务:
```bash
cat > ~/gdrive/DLQI/tasks/train_001.json << 'EOF'
{
    "task_id": "train_001",
    "type": "train",
    "symbol": "600519",
    "model_type": "lstm",
    "start_date": "2020-01-01",
    "epochs": 50
}
EOF
```

In [ ]:
# ===== 一键提交任务 =====
# 修改下面的参数，运行此 cell 即可提交任务

def submit_task(task: dict):
    """往 tasks/ 目录写入任务 JSON"""
    tid = task.get('task_id', f"task_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    task['task_id'] = tid
    path = os.path.join(TASKS_DIR, f"{tid}.json")
    with open(path, 'w') as f:
        json.dump(task, f, indent=2)
    print(f"✅ 任务已提交: {path}")
    print(f"   内容: {json.dumps(task, ensure_ascii=False)}")
    return tid

# ---- 选一个取消注释运行 ----

# 方式1: 先下载数据，再联合训练（推荐首次使用）
submit_task({"type": "download_cn", "start_date": "2016-01-01"})
submit_task({"type": "train_multi", "epochs": 30})

# 方式2: 只联合训练（数据已下载过）
# submit_task({"type": "train_multi", "epochs": 30})

# 方式3: 单股票训练
# submit_task({"type": "train", "symbol": "600519", "model_type": "transformer", "epochs": 50})

In [ ]:
processor.run_loop(interval=30)

---\n## 手动训练 (可选)\n\n如果不想用任务队列, 直接在下面运行:

In [ ]:
# 手动训练 - 修改这里的参数后运行此单元格
SYMBOL = "600519"
MODEL_TYPE = "lstm"       # lstm / transformer / lightgbm / xgboost
START_DATE = "2016-01-01"
EPOCHS = 50

print(f"\U0001f4ca 训练 {MODEL_TYPE} | {SYMBOL} | {START_DATE}~")
df = trainer.fetch_data(SYMBOL, START_DATE)
print(f"   数据: {len(df)} 条")
X, y, features = trainer.prepare_features(df)
print(f"   特征: {X.shape}")

if MODEL_TYPE == 'lstm': model, loss = trainer.train_lstm(X, y, epochs=EPOCHS)
elif MODEL_TYPE == 'transformer': model, loss = trainer.train_transformer(X, y, epochs=EPOCHS)
elif MODEL_TYPE == 'lightgbm': model, loss = trainer.train_lightgbm(X, y)
elif MODEL_TYPE == 'xgboost': model, loss = trainer.train_xgboost(X, y)

mid = trainer.save_model(model, MODEL_TYPE, SYMBOL, {'val_loss': float(loss), 'epochs': EPOCHS})
print(f"\n\u2705 完成! 模型ID: {mid}, Val Loss: {loss:.6f}")